In [2]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import copy

# -------------------------------
# DATASET DEFINITION
# -------------------------------
class UCFCrimeDataset(Dataset):
    def __init__(self, base_dir):
        self.files = []
        self.labels = []
        categories = sorted(os.listdir(base_dir))
        self.label_map = {cat: i for i, cat in enumerate(categories)}
        print("Classes mapping:", self.label_map)
        
        for cat in categories:
            cat_path = os.path.join(base_dir, cat)
            if not os.path.isdir(cat_path):
                continue
            for f in os.listdir(cat_path):
                if f.endswith('.npy'):
                    self.files.append(os.path.join(cat_path, f))
                    self.labels.append(self.label_map[cat])
        print(f"Total samples found: {len(self.files)}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        x = np.load(self.files[idx])  
        y = self.labels[idx]
        return torch.from_numpy(x).float(), torch.tensor(y, dtype=torch.long)

# -------------------------------
# TRANSFORMER ARCHITECTURE
# -------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0)) 

    def forward(self, x):
        # x shape: [Batch, Seq_Len, d_model]
        x = x + self.pe[:, :x.size(1), :]
        return x

class CrimeTransformer(nn.Module):
    def __init__(self, input_dim=512, num_classes=14, num_heads=8, num_layers=4, forward_expansion=4, dropout=0.3):
        super().__init__()
        
        self.pos_encoder = PositionalEncoding(d_model=input_dim)
        
        # Standard Transformer Encoder Layer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_dim, 
            nhead=num_heads, 
            dim_feedforward=input_dim * forward_expansion, 
            dropout=dropout, 
            batch_first=True,
            activation='gelu' # GELU generally performs better than ReLU in transformers
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x: [Batch, Sequence_Length, 512]
        x = self.pos_encoder(x)
        encoded = self.transformer_encoder(x)
        
        # Global Average Pooling across the sequence (time) dimension
        # Averages all the frame features into one comprehensive video representation
        pooled = encoded.mean(dim=1)  
        
        return self.classifier(pooled)

# -------------------------------
# EXECUTION SCRIPT
# -------------------------------
def train_transformer():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing Transformer on: {device}")
    
    DATA_PATH = "./extracted_features" 
    dataset = UCFCrimeDataset(DATA_PATH)
    
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    # FIX: num_workers set to 0 to prevent Windows process crashing
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

    num_classes = len(dataset.label_map)
    # Instantiate the Transformer
    model = CrimeTransformer(input_dim=512, num_classes=num_classes).to(device)
    
    # Label Smoothing Cross Entropy helps transformers generalize better
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1) 
    
    # Transformers are highly sensitive to learning rates. AdamW with a lower LR is standard.
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4) 
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

    EPOCHS = 100
    best_val_loss = float('inf')
    
    # Ensure the save directory exists to prevent FileNotFoundError
    os.makedirs("TRANSFORMER", exist_ok=True)
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_loader)

        # Validation Phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                outputs = model(x)
                loss = criterion(outputs, y)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += y.size(0)
                correct += (predicted == y).sum().item()
                
        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100 * correct / total
        
        scheduler.step() # Step CosineAnnealing every epoch
        
        current_lr = scheduler.get_last_lr()[0]
        print(f"Epoch [{epoch+1}/{EPOCHS}] | LR: {current_lr:.6f} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "TRANSFORMER/best_transformer_model.pth")
            print("  --> Transformer model saved!")

    np.save("TRANSFORMER/transformer_label_map.npy", dataset.label_map)
    print("Transformer Training Complete.")

if __name__ == "__main__":
    train_transformer()

Executing Transformer on: cuda
Classes mapping: {'Abuse': 0, 'Arrest': 1, 'Arson': 2, 'Assault': 3, 'Burglary': 4, 'Explosion': 5, 'Fighting': 6, 'NormalVideos': 7, 'RoadAccidents': 8, 'Robbery': 9, 'Shooting': 10, 'Shoplifting': 11, 'Stealing': 12, 'Vandalism': 13}
Total samples found: 1610
Epoch [1/100] | LR: 0.000300 | Train Loss: 2.0406 | Val Loss: 1.9508 | Val Acc: 48.45%
  --> Transformer model saved!
Epoch [2/100] | LR: 0.000299 | Train Loss: 1.8625 | Val Loss: 1.6854 | Val Acc: 57.45%
  --> Transformer model saved!
Epoch [3/100] | LR: 0.000297 | Train Loss: 1.7369 | Val Loss: 1.7198 | Val Acc: 54.66%
Epoch [4/100] | LR: 0.000295 | Train Loss: 1.6273 | Val Loss: 1.6377 | Val Acc: 57.14%
  --> Transformer model saved!
Epoch [5/100] | LR: 0.000293 | Train Loss: 1.5481 | Val Loss: 1.7164 | Val Acc: 52.80%
Epoch [6/100] | LR: 0.000290 | Train Loss: 1.4448 | Val Loss: 1.6736 | Val Acc: 57.14%
Epoch [7/100] | LR: 0.000286 | Train Loss: 1.4126 | Val Loss: 1.8025 | Val Acc: 52.80%
Epoch

In [3]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# -------------------------------
# DATASET DEFINITION
# -------------------------------
class UCFCrimeDataset(Dataset):
    def __init__(self, base_dir):
        self.files = []
        self.labels = []
        
        if os.path.exists(base_dir) and os.path.isdir(base_dir):
            categories = sorted(os.listdir(base_dir))
            self.label_map = {cat: i for i, cat in enumerate(categories)}
            
            for cat in categories:
                cat_path = os.path.join(base_dir, cat)
                if not os.path.isdir(cat_path):
                    continue
                for f in os.listdir(cat_path):
                    if f.endswith('.npy'):
                        self.files.append(os.path.join(cat_path, f))
                        self.labels.append(self.label_map[cat])
        else:
            classes = ["Abuse", "Arrest", "Arson", "Assault", "Burglary", "Explosion", 
                       "Fighting", "Normal", "RoadAccidents", "Robbery", "Shooting", 
                       "Shoplifting", "Stealing", "Vandalism"]
            self.label_map = {cat: i for i, cat in enumerate(classes)}
            print(f"Warning: Directory '{base_dir}' missing. Using template mapping.")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        x = np.load(self.files[idx])  
        y = self.labels[idx]
        return torch.from_numpy(x).float(), torch.tensor(y, dtype=torch.long)

# -------------------------------
# TRANSFORMER ARCHITECTURE
# -------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0)) 

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

class CrimeTransformer(nn.Module):
    def __init__(self, input_dim=512, num_classes=14, num_heads=8, num_layers=4, forward_expansion=4, dropout=0.3):
        super().__init__()
        self.pos_encoder = PositionalEncoding(d_model=input_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_dim, nhead=num_heads, 
            dim_feedforward=input_dim * forward_expansion, 
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.pos_encoder(x)
        encoded = self.transformer_encoder(x)
        pooled = encoded.mean(dim=1)  
        return self.classifier(pooled)

# -------------------------------
# EVALUATION SCRIPT
# -------------------------------
def evaluate_transformer():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Testing on device: {device}")
    
    DATA_PATH = "./extracted_features"
    MODEL_PATH = "TRANSFORMER/best_transformer_model.pth"
    
    dataset = UCFCrimeDataset(DATA_PATH)
    inverse_label_map = {v: k for k, v in dataset.label_map.items()}
    num_classes = len(dataset.label_map)
    
    model = CrimeTransformer(input_dim=512, num_classes=num_classes).to(device)
    
    if os.path.exists(MODEL_PATH):
        model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
        print(f"Loaded weights from {MODEL_PATH}")
    else:
        print(f"Error: {MODEL_PATH} not found. Please train the model first.")
        return

    # num_workers=0 prevents Windows multiprocessing crashes
    eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)
    
    model.eval()
    all_preds = []
    all_labels = []
    
    print("\nRunning Evaluation... (This may take a moment)")
    with torch.no_grad():
        for x, y in eval_loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    print(f"\n==================================================")
    print(f"Transformer Overall Accuracy: {accuracy * 100:.2f}%")
    print(f"==================================================")
    
    print("\nDetailed Classification Report:")
    target_names = [inverse_label_map[i] for i in range(len(inverse_label_map))]
    print(classification_report(all_labels, all_preds, target_names=target_names, zero_division=0))
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))

if __name__ == "__main__":
    evaluate_transformer()

Testing on device: cuda
Loaded weights from TRANSFORMER/best_transformer_model.pth

Running Evaluation... (This may take a moment)

Transformer Overall Accuracy: 60.62%

Detailed Classification Report:
               precision    recall  f1-score   support

        Abuse       0.00      0.00      0.00        48
       Arrest       0.13      0.04      0.07        45
        Arson       0.67      0.05      0.09        41
      Assault       0.15      0.04      0.07        47
     Burglary       0.24      0.31      0.27        87
    Explosion       0.00      0.00      0.00        29
     Fighting       0.28      0.11      0.16        45
 NormalVideos       0.75      0.94      0.83       800
RoadAccidents       0.46      0.64      0.53       127
      Robbery       0.44      0.28      0.34       145
     Shooting       0.00      0.00      0.00        27
  Shoplifting       0.00      0.00      0.00        29
     Stealing       0.37      0.68      0.48        95
    Vandalism       0.00   

In [4]:
import os
import cv2
import math
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from collections import deque

# -------------------------------
# TRANSFORMER ARCHITECTURE
# -------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0)) 

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

class CrimeTransformer(nn.Module):
    def __init__(self, input_dim=512, num_classes=14, num_heads=8, num_layers=4, forward_expansion=4, dropout=0.3):
        super().__init__()
        self.pos_encoder = PositionalEncoding(d_model=input_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_dim, nhead=num_heads, 
            dim_feedforward=input_dim * forward_expansion, 
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.pos_encoder(x)
        encoded = self.transformer_encoder(x)
        pooled = encoded.mean(dim=1)  
        return self.classifier(pooled)

# -------------------------------
# LIVE INFERENCE ENGINE
# -------------------------------
def run_transformer_live():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Initializing criDe Transformer engine on: {device}")

    classes = ["Abuse", "Arrest", "Arson", "Assault", "Burglary", "Explosion", 
               "Fighting", "Normal", "RoadAccidents", "Robbery", "Shooting", 
               "Shoplifting", "Stealing", "Vandalism"]
    
    num_classes = len(classes)
    model = CrimeTransformer(input_dim=512, num_classes=num_classes).to(device)
    
    MODEL_PATH = "TRANSFORMER/best_transformer_model.pth"
    if os.path.exists(MODEL_PATH):
        model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
        print("Transformer weights loaded successfully.")
    else:
        print("Warning: Weights not found. Running with random initialization.")
    model.eval()

    # Pre-trained Feature Extractor (Outputs 512-dim vectors per frame)
    print("Loading ResNet18 Feature Extractor...")
    resnet = models.resnet18(pretrained=True)
    resnet.fc = nn.Identity() 
    resnet = resnet.to(device)
    resnet.eval()

    preprocess = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    SEQUENCE_LENGTH = 32  # Number of frames required for context
    feature_buffer = deque(maxlen=SEQUENCE_LENGTH)

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    print("Live stream active. Press 'q' to exit.")
    current_prediction = "Buffering..."
    confidence_text = ""

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        input_frame = frame.copy()
        tensor_frame = preprocess(input_frame).unsqueeze(0).to(device)
        
        with torch.no_grad():
            frame_features = resnet(tensor_frame).squeeze(0).cpu().numpy()
        
        feature_buffer.append(frame_features)

        if len(feature_buffer) == SEQUENCE_LENGTH:
            sequence_tensor = torch.tensor(np.array(feature_buffer), dtype=torch.float).unsqueeze(0).to(device)
            
            with torch.no_grad():
                raw_logits = model(sequence_tensor)
                probabilities = torch.softmax(raw_logits, dim=1)
                confidence, predicted_idx = torch.max(probabilities, 1)
                
            current_prediction = classes[predicted_idx.item()]
            confidence_text = f"{confidence.item() * 100:.1f}%"

        text_color = (0, 255, 0) if current_prediction == "Normal" or "Buffering" in current_prediction else (0, 0, 255)
        display_string = f"Action: {current_prediction} {confidence_text}"
        
        cv2.putText(frame, display_string, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, text_color, 2, cv2.LINE_AA)
        cv2.imshow("criDe - Transformer Live", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_transformer_live()

Initializing criDe Transformer engine on: cuda
Transformer weights loaded successfully.
Loading ResNet18 Feature Extractor...


C:\Users\navee\AppData\Roaming\Python\Python314\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\navee\AppData\Roaming\Python\Python314\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Live stream active. Press 'q' to exit.
